# Unconstrained sampling with $J_s(x)=s\,[x]_\times$

The same study as `Constrained_Sampling_ANDS_ball_constrained.ipynb`, with the ball
$K=\{\lVert x\rVert_2\le R\}$ removed. Same field

$$J_s(x)=\begin{pmatrix}0&-s x_3& s x_2\\ s x_3&0&-s x_1\\ -s x_2& s x_1&0\end{pmatrix},
\qquad J_s(x)\,w=s\,(x\times w),$$

same two targets, same $J=0$ against $s=8,16$, same $W_1$-to-floor methodology and the same
8 replicas. The update loses its projection and becomes

$$x\ \leftarrow\ x-\eta\,e^{\Delta}\bigl(I+J_s(x)\bigr)\nabla U_0+\sqrt{2\eta}\,e^{\Delta/2}\xi .$$

## Three things the constraint was hiding

**The target is 10× wider and the run has to be 16× longer.** Truncating at $R=0.5$ left a target
that was nearly a uniform ball and converged in a few hundred iterations. Untruncated, the
coordinate standard deviations are $(1.01,\,2.02,\,4.05)$ for Target A, the 99th percentile of
$\lVert x\rVert$ is $13.1$, and at the original `MAXIT = 5000` the reversible chain is still at
**12.9×** the sampling floor on A and **8.8×** on B — not converged, nowhere near it. A pilot run is
reported below; `MAXIT` is raised to 80 000 accordingly. Keeping 5000 would not have been "the same
experiment", it would have been a measurement of the transient.

**The anisotropy comes back.** The constrained notebook recorded anisotropy 1.18 and 1.32 against
"34.6 unconstrained" and noted that any gain on the ball therefore came from constraint geometry
alone. Two different statistics were being compared there — 34.6 is the condition number of $\Sigma$,
while 1.18 is a ratio of coordinate standard deviations. Reported consistently, the coordinate-sd
anisotropy here is **4.00** and **4.11** against 1.18 and 1.32 on the ball, and the condition number
of $\Sigma$ is 34.6 either way since truncation does not change $\Sigma$. Both statistics are given
below so the comparison is like for like.

**Two of the field's three properties survive; one becomes vacuous.** Antisymmetry and
$\nabla\!\cdot J_s=0$ are algebraic identities and hold unchanged, so the update still carries no
correction term. Wall-tangency has no wall. But the identity behind it, $x^\top J_s(x)w=0$, is still
true and still does something: the skew drift is tangential to *every* sphere about the origin, so it
preserves $\lVert x\rVert$ to first order. Experiment 4 is about what it does at **second** order,
which is where the boundary atom went.

In [ ]:
%matplotlib inline
import math, os, time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy.stats import ks_2samp, wasserstein_distance

torch.set_default_dtype(torch.float64)
torch.set_num_threads(4)
os.makedirs("figures", exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")

In [ ]:
# -----------------------------
# Settings
# -----------------------------
d        = 3
N        = 5000         # walkers per replica
REPLICAS = 8            # independent replicas, run as one batched ensemble
MAXIT    = 80000        # raised from 5000: see the pilot below.  Without the wall the
                        # reversible chain needs ~60k iterations, not ~500.
eta      = 5e-4
delta    = 0.1          # anchor smoothing
x_start  = 0.5
SEED     = 106
TRACK    = 400          # 200 tracking points, as before (was 25 for MAXIT 5000)
S_LIST   = [8.0, 16.0]
S_TOP    = 16.0         # strongest field kept; used for the stepsize study

SCALES, RHO = [0.5, 1.0, 2.0], 0.6
def make_L(dim, scales, rho):
    Dm = np.diag(scales)
    Cm = np.full((dim, dim), rho); np.fill_diagonal(Cm, 1.0)
    return np.linalg.cholesky(Dm @ Cm @ Dm)

L3   = make_L(d, SCALES, RHO)
SIG  = L3 @ L3.T
Minv = torch.tensor(np.linalg.inv(L3))
EVAL, EVEC = np.linalg.eigh(SIG)
print("Sigma =\n", np.round(SIG, 3))
print("eigenvalues", np.round(EVAL, 3), "  condition number {:.1f}".format(EVAL[-1] / EVAL[0]))
print()
print("Sigma is unchanged by the truncation -- the constraint reshaped the target, not the")
print("covariance that defines it.  What changes is how much of that shape the sampler sees.")

## The field, and which of its conditions still mean anything

`Js_matrix` builds the specified matrix entry by entry; `Js` is the matrix-free $s\,(x\times w)$ used
in the sampler. The first check is that they agree.

Of the three properties the constrained notebook verified, **antisymmetry** and
**$\nabla\!\cdot J_s=0$** are algebraic and survive untouched — the second is still what lets the
update drop the correction term $\nabla\!\cdot J$ that a plane-aimed field would need at every step.
**Wall-tangency** no longer has a wall, so in its place the table checks the identity it came from,
$x^\top J_s(x)w=0$, which now says the skew drift is tangential to every sphere about the origin.

In [ ]:
def Js_matrix(s, x):
    # the specified matrix, assembled entry by entry:
    #   [[0, -s x3,  s x2], [s x3, 0, -s x1], [-s x2, s x1, 0]]
    z = torch.zeros(len(x))
    return torch.stack([
        torch.stack([        z, -s * x[:, 2],  s * x[:, 1]], 1),
        torch.stack([ s * x[:, 2],         z, -s * x[:, 0]], 1),
        torch.stack([-s * x[:, 1],  s * x[:, 0],         z], 1)], 1)


@torch.no_grad()
def Js(s, x, g):
    # J_s(x) g = s (x cross g).  s = 0 gives the reversible scheme.
    return torch.zeros_like(g) if s == 0.0 else s * torch.cross(x, g, dim=1)


def autograd_div(s, x):
    # (div J)_i = sum_j d_j J_ij, by autograd, using no closed form
    n = x.shape[0]
    x = x.detach().clone().requires_grad_(True)
    out = torch.zeros(n, d)
    eye = torch.eye(d)
    for j in range(d):
        col = s * torch.cross(x, eye[j].expand(n, d), dim=1)
        for i in range(d):
            gi = torch.autograd.grad(col[:, i].sum(), x, retain_graph=True, allow_unused=True)[0]
            if gi is not None:
                out[:, i] += gi[:, j]
    return out.detach()


gen = torch.Generator().manual_seed(0)
# sample over the radial range the UNCONSTRAINED target actually occupies, not a unit ball
xs = 4.0 * torch.randn(400, d, generator=gen)
gs = torch.randn(400, d, generator=gen)

rows = []
for s in S_LIST:
    M = Js_matrix(s, xs)
    rows.append({
        "s": s,
        "matrix form vs s(x x g)": "{:.1e}".format(float(
            (torch.einsum("nij,nj->ni", M, gs) - Js(s, xs, gs)).abs().max())),
        "max |J + J^T|": "{:.1e}".format(float((M + M.transpose(1, 2)).abs().max())),
        "max |div J| (autograd)": "{:.1e}".format(float(autograd_div(s, xs[:40]).abs().max())),
        "max |x.J(x)g| / |x||g|": "{:.1e}".format(float(
            ((Js(s, xs, gs) * xs).sum(1).abs()
             / (xs.norm(dim=1) * gs.norm(dim=1) * s)).max())),
        "||J||_2 vs s||x||": "{:.1e}".format(float(
            (torch.linalg.matrix_norm(M, ord=2) - s * xs.norm(dim=1)).abs().max()))})
print("Antisymmetric and divergence-free exactly, at every strength, as on the ball -- the")
print("divergence column is what lets the correction term stay dropped.  The fourth column")
print("replaces the old wall-tangency check: the skew drift is orthogonal to x everywhere, so")
print("it is tangential to every sphere about the origin and preserves ||x|| to FIRST order.")
print("Experiment 4 measures what happens at second order.\n")
pd.DataFrame(rows).set_index("s")

In [ ]:
# the geometry the field cannot change: rotation plane is x-perp, strength grows with ||x||.
# On the ball this was bounded by sR.  Unconstrained it is not bounded at all.
rr = np.linspace(0, 14, 200)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
for s in S_LIST:
    axes[0].plot(rr, s * rr, linewidth=2, label="s = {:g}".format(s))
axes[0].axvline(0.5, color="k", linestyle="--", linewidth=1.3,
                label="R = 0.5 (old ceiling)")
axes[0].set_xlabel(r"$\|x\|_2$", fontsize=12)
axes[0].set_ylabel(r"$\|J_s(x)\|_2 = s\|x\|$", fontsize=12)
axes[0].set_yscale("log")
axes[0].set_title("unbounded now: the ball capped the field at $sR$", fontsize=11)
axes[0].grid(alpha=0.3); axes[0].legend(fontsize=9)

cos = (Js(8.0, xs, gs) * xs).sum(1) / (Js(8.0, xs, gs).norm(dim=1) * xs.norm(dim=1))
axes[1].hist(cos.numpy(), bins=40, color="#4C72B0")
axes[1].set_xlabel(r"$\cos\angle\,(J_s(x)g,\ x)$", fontsize=12)
axes[1].set_ylabel("count", fontsize=12)
axes[1].set_xlim(-1, 1)
axes[1].set_title(r"the skew drift is always $\perp x$: the plane is fixed by position", fontsize=11)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/uncon_cross_field_geometry.pdf", dpi=300, bbox_inches="tight")
plt.show()
print("max |cos| = {:.1e}   -> the rotation plane is x-perp everywhere and cannot be aimed;".format(
    float(cos.abs().max())))
print("the only free parameter is still s.  But sR was the natural strength scale on the ball,")
print("and without a wall there is no R: the effective strength a walker feels is s||x||, which")
print("varies across the ensemble by more than an order of magnitude.")

In [ ]:
# -----------------------------
# Result cache and robust density panels
# -----------------------------
# Every expensive block below writes its result to cache/ and reloads it on a re-run.
# The first attempt at this notebook lost 45 minutes of completed sampling because a
# plotting cell raised and `nbconvert --inplace` only writes the notebook at the end.
import pickle

CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def cached(name, fn, force=False):
    p = os.path.join(CACHE_DIR, name + ".pkl")
    if os.path.exists(p) and not force:
        with open(p, "rb") as fh:
            return pickle.load(fh)
    out = fn()
    with open(p, "wb") as fh:
        pickle.dump(out, fh)
    return out


def density_panel(ax, X, lim, cmap="mako_r", levels=9, thresh=0.02):
    '''
    Filled density contours for a 2-d marginal, with two guards the constrained
    notebook did not need:

      * points outside the plotted window are dropped before the estimate, so a
        handful of far-tail walkers cannot stretch the KDE grid until essentially
        all the density lands in one cell.  The fraction kept is returned.
      * seaborn raises "Contour levels must be increasing" when the density is
        peaked enough that its quantile-spaced levels collide -- the l1 cusp at
        the origin does this.  On that failure the panel falls back to a 2-d
        histogram, which cannot degenerate and does not smooth the cusp away.
    '''
    inside = (np.abs(X[:, 0]) <= lim) & (np.abs(X[:, 1]) <= lim)
    Xi = X[inside]
    frac = float(inside.mean())
    try:
        sns.kdeplot(x=Xi[:, 0], y=Xi[:, 1], fill=True, levels=levels, ax=ax,
                    thresh=thresh, cmap=cmap)
    except ValueError:
        ax.hist2d(Xi[:, 0], Xi[:, 1], bins=80,
                  range=[[-lim, lim], [-lim, lim]], cmap=cmap)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    return frac

In [ ]:
# -----------------------------
# Targets, exact reference samples, and the sampler
# -----------------------------
def build_target(kind):
    if kind == "ell":
        @torch.no_grad()
        def U(x):  return torch.norm(x @ Minv.T, dim=1)
        @torch.no_grad()
        def U0(x): return torch.sqrt(torch.sum((x @ Minv.T) ** 2, dim=1) + delta ** 2)
        @torch.no_grad()
        def gU0(x):
            y = x @ Minv.T
            return (y / torch.sqrt(torch.sum(y ** 2, dim=1, keepdim=True) + delta ** 2)) @ Minv
    else:
        @torch.no_grad()
        def U(x):  return torch.sum(torch.abs(x @ Minv.T), dim=1)
        @torch.no_grad()
        def U0(x): return torch.sum(torch.sqrt((x @ Minv.T) ** 2 + delta ** 2), dim=1)
        @torch.no_grad()
        def gU0(x):
            y = x @ Minv.T
            return (y / torch.sqrt(y ** 2 + delta ** 2)) @ Minv
    return U, U0, gU0


def sample_target(kind, n, rng):
    # Exact i.i.d. draws.  On the ball this needed rejection; here the draw IS the target,
    # so the reference is exact at no cost and with no acceptance rate to worry about.
    if kind == "ell":
        r = rng.gamma(d, 1.0, n)                      # radial law of e^{-|y|} in d = 3
        u = rng.normal(size=(n, d)); u /= np.linalg.norm(u, axis=1, keepdims=True)
        y = r[:, None] * u
    else:
        y = rng.laplace(0.0, 1.0, size=(n, d))
    return y @ L3.T


def w1_floor(ref, kind, rng, reps=4):
    return float(np.mean([[wasserstein_distance(ref[:, i], sample_target(kind, len(ref), rng)[:, i])
                           for i in range(d)] for _ in range(reps)]))


def radial_stats(X):
    # replaces on_boundary(): there is no boundary, so the quantity of interest is whether
    # the cloud has the right radial extent
    r = np.linalg.norm(X, axis=1)
    return {"mean_r": float(r.mean()), "q99_r": float(np.quantile(r, 0.99)),
            "sd": X.std(0)}


@torch.no_grad()
def run_replicas(kind_target, s, ref, S=REPLICAS, n_steps=MAXIT, seed=SEED, eta=eta,
                 track=TRACK, x0=None):
    # Anchored Langevin with the J_s drift, no projection.
    # div J_s = 0 exactly, so there is still NO correction term in this update.
    U, U0, gU0 = build_target(kind_target)
    torch.manual_seed(seed)
    x = torch.full((S * N, d), x_start) if x0 is None else torch.tensor(x0).clone()
    it, W, Wc = [], [], []
    for k in range(n_steps):
        Delta = U(x) - U0(x)
        g = gU0(x) * torch.exp(Delta).unsqueeze(1)
        x = (x - eta * (g + Js(s, x, g))
             + np.sqrt(2 * eta) * torch.exp(0.5 * Delta).unsqueeze(1) * torch.randn_like(x))
        if not torch.isfinite(x).all():
            # the skew field can drive |x| past the escape radius of Experiment 4b; when it
            # overflows there is nothing left to measure, so stop and say so rather than
            # letting a NaN take out the rest of the notebook
            print("      !! non-finite state at iteration {} (s = {:g}) -- chain diverged"
                  .format(k + 1, s), flush=True)
            break
        if (k + 1) % track == 0:
            xn = x.numpy().reshape(S, N, d)
            per = np.array([[wasserstein_distance(ref[:, i], xn[r, :, i]) for i in range(d)]
                            for r in range(S)])
            it.append(k + 1)
            W.append(per.mean(axis=1))
            Wc.append(per.mean(axis=0))
    return np.array(it), np.array(W).T, np.array(Wc), x.numpy().reshape(S, N, d)


def tau_one(it, w, thr):
    bad = np.where(w > thr)[0]
    j = 0 if len(bad) == 0 else bad[-1] + 1
    return float(it[j]) if j < len(it) else np.nan


def boot(v, B=4000, seed=0):
    v = np.asarray(v, float)
    if not np.any(np.isfinite(v)):
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    draws = [np.nanmean(x) for x in (v[rng.integers(0, len(v), len(v))] for _ in range(B))
             if np.any(np.isfinite(x))]
    return np.nanmean(v), float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))


def boot_speedup(base, cur, B=4000, seed=1):
    base, cur = np.asarray(base, float), np.asarray(cur, float)
    if not (np.any(np.isfinite(base)) and np.any(np.isfinite(cur))):
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    draws = []
    for _ in range(B):
        a, b = base[rng.integers(0, len(base), len(base))], cur[rng.integers(0, len(cur), len(cur))]
        if np.any(np.isfinite(a)) and np.any(np.isfinite(b)):
            draws.append(np.nanmean(a) / np.nanmean(b))
    return (np.nanmean(base) / np.nanmean(cur),
            float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5)))


def fmt_tau(t):
    return "never" if not np.isfinite(t[0]) else "{:.0f} [{:.0f}, {:.0f}]".format(*t)


def fmt_speed(sp):
    return "never reaches it" if not np.isfinite(sp[0]) else "{:.2f}x [{:.2f}, {:.2f}]".format(*sp)

### Pilot: how long does the unconstrained problem actually need?

The constrained study used `MAXIT = 5000`, which was generous on a ball of radius 0.5. This cell
checks what that horizon buys once the wall is gone, before committing 80 000 iterations to every
run. It is cheap — two replicas of 2000 walkers, $J=0$ only.

In [ ]:
_rng = np.random.default_rng(SEED)
print("pilot: J = 0, 2 x 2000 walkers, eta = {:.0e}\n".format(eta))
print("{:>8}   {:>10}   {:>16}   {:>16}".format("iter", "target", "W1", "multiple of floor"))
_pilot = {}
for _kind in ["ell", "l1"]:
    _ref = sample_target(_kind, 5000, _rng)
    _floor = w1_floor(_ref, _kind, _rng)
    _U, _U0, _gU0 = build_target(_kind)
    torch.manual_seed(SEED)
    _x = torch.full((2 * 2000, d), x_start)
    _rec = []
    with torch.no_grad():
        for _k in range(1, 60001):
            _D = _U(_x) - _U0(_x)
            _g = _gU0(_x) * torch.exp(_D).unsqueeze(1)
            _x = (_x - eta * _g
                  + np.sqrt(2 * eta) * torch.exp(0.5 * _D).unsqueeze(1) * torch.randn_like(_x))
            if _k in (5000, 10000, 20000, 40000, 60000):
                _xn = _x.numpy().reshape(2, 2000, d)
                _w = np.mean([[wasserstein_distance(_ref[:, i], _xn[r, :, i]) for i in range(d)]
                              for r in range(2)])
                _rec.append((_k, _w, _w / _floor))
                print("{:>8d}   {:>10}   {:>16.4f}   {:>15.2f}x".format(_k, _kind, _w, _w / _floor))
    _pilot[_kind] = _rec
print()
print("At the original MAXIT = 5000 the reversible chain sits at {:.1f}x the floor on ell and".format(
    _pilot["ell"][0][2]))
print("{:.1f}x on l1.  That is the transient, not the stationary law.  MAXIT = 80000 below.".format(
    _pilot["l1"][0][2]))
print("(The pilot uses 2000 walkers against a 5000-sample reference, so its W1 carries extra")
print(" sampling noise and slightly overstates the floor multiple; the runs below use N = 5000.)")

## Runs

$\tau$ is the iteration at which the mean $W_1$ reaches a given multiple of the sampling floor and
stays there — the floor being $W_1$ between two independent exact draws, which no sampler can beat.
Bands and intervals are over 8 independent replicas.

In [ ]:
TARGETS = {"ell": "Target A: elliptical Laplace on $\\mathbb{R}^3$",
           "l1":  "Target B: correlated $\\ell_1$ Laplace on $\\mathbb{R}^3$"}
GRID = [(0.0, "J = 0")] + [(s, "$J_s$, s={:g}".format(s)) for s in S_LIST]

# tau is read at several multiples of the floor.  A threshold close to a scheme's own
# stationary error turns tau into a noise statistic (the curve keeps re-crossing it), so
# each entry is reported only where the scheme actually settles below the threshold.
MULT = (2.0, 3.0, 5.0)

rng = np.random.default_rng(SEED)
refs, floors, RES = {}, {}, {}
for kind in TARGETS:
    refs[kind] = sample_target(kind, N, rng)
    floors[kind] = w1_floor(refs[kind], kind, rng)
    RES[kind] = {}
    rs = radial_stats(refs[kind])
    print("\n### {}   floor {:.4f}   mean ||x|| {:.3f}   coord sd {}".format(
        kind, floors[kind], rs["mean_r"], np.round(rs["sd"], 3)), flush=True)
    for s, lab in GRID:
        t0 = time.time()
        def _go(kind=kind, s=s):
            it, W, Wc, X = run_replicas(kind, s, refs[kind])
            return {"it": it, "W": W, "Wc": Wc, "X": X, "s": s,
                    "taus": {m: np.array([tau_one(it, w, m * floors[kind]) for w in W])
                             for m in MULT},
                    "stat": W[:, -10:].mean(axis=1),
                    "rad": radial_stats(X.reshape(-1, d))}
        RES[kind][lab] = cached("main_{}_{:g}".format(kind, s), _go)
        print("   {:<20} {:5.0f}s".format(lab.replace("$", ""), time.time() - t0), flush=True)

MAIN = [lab for _, lab in GRID]
COL = {"J = 0": "#C44E52", "$J_s$, s=8": "#4C72B0", "$J_s$, s=16": "#55A868"}
TOP = "$J_s$, s={:g}".format(S_TOP)

### What the target looks like without the truncation

The constrained notebook asked whether the target filled $K$, since a constrained experiment is only
about the constraint if mass reaches the wall. There is no wall here, so the corresponding question is
how much shape the sampler now has to reproduce — radial extent, and anisotropy reported two ways so
the comparison with the ball is like for like.

In [ ]:
BALL = {"ell": {"aniso": 1.18, "med": 0.762}, "l1": {"aniso": 1.32, "med": 0.737}}
rows = []
for kind in TARGETS:
    r = np.linalg.norm(refs[kind], axis=1)
    sd = refs[kind].std(0)
    rows.append({"target": kind,
                 "mean ||x||": round(float(r.mean()), 3),
                 "median ||x||": round(float(np.median(r)), 3),
                 "q99 ||x||": round(float(np.quantile(r, 0.99)), 3),
                 "coordinate sds": np.round(sd, 3),
                 "anisotropy sd_max/sd_min": round(float(sd.max() / sd.min()), 2),
                 "  same, on the ball": BALL[kind]["aniso"],
                 "cond(Sigma)": round(float(EVAL[-1] / EVAL[0]), 1)})
display(pd.DataFrame(rows).set_index("target"))
print("Anisotropy is 4.00 and 4.11 against 1.18 and 1.32 on the ball -- truncation at R = 0.5 was")
print("flattening the target towards a uniform sphere.  cond(Sigma) = 34.6 is the same in both")
print("settings, because truncation does not change Sigma; the constrained notebook compared that")
print("number against a coordinate-sd ratio, which is why the contrast looked larger there.")
print()
print("The radial extent is the substantive change: q99 ||x|| is ~{:.0f}x the old ball radius, and".format(
    np.quantile(np.linalg.norm(refs['ell'], axis=1), 0.99) / 0.5))
print("since ||J_s(x)|| = s||x||, walkers in the tail feel a field ~{:.0f}x stronger than the".format(
    np.quantile(np.linalg.norm(refs['ell'], axis=1), 0.99) / np.median(np.linalg.norm(refs['ell'], axis=1))))
print("median walker.  On the ball that spread was capped at sR for everyone.")

## Experiment 1 — $W_1$ to the target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
for ax, kind in zip(axes, TARGETS):
    for lab in MAIN:
        r = RES[kind][lab]
        ax.plot(r["it"], r["W"].mean(axis=0), linewidth=2.2, color=COL[lab], label=lab)
        ax.fill_between(r["it"], np.percentile(r["W"], 5, axis=0),
                        np.percentile(r["W"], 95, axis=0), color=COL[lab], alpha=0.16, linewidth=0)
    ax.axhline(floors[kind], color="k", linestyle=":", linewidth=1.7, label="sampling floor")
    ax.axhline(2 * floors[kind], color="0.5", linestyle="--", linewidth=1.2,
               label=r"$2\times$ floor")
    ax.axvline(5000, color="0.3", linestyle="-.", linewidth=1.2,
               label="old MAXIT (5000)")
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(200, MAXIT)
    ax.set_xlabel("Iterations", fontsize=13)
    ax.set_ylabel(r"mean $W_1$   ({} replicas, 5-95%)".format(REPLICAS), fontsize=12)
    ax.set_title(TARGETS[kind], fontsize=12)
    ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=9)
fig.suptitle(r"Convergence to the unconstrained $\pi$, with $J_s(x)=s\,[x]_\times$", fontsize=14)
plt.tight_layout()
plt.savefig("figures/uncon_cross_w1.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# per-coordinate: the three coordinates have sds of roughly 1, 2 and 4, so the mean hides
# which direction the rotation is helping.  On the ball these were nearly equal.
for kind in TARGETS:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.3), sharey=True)
    for i, ax in enumerate(axes):
        for lab in MAIN:
            r = RES[kind][lab]
            ax.plot(r["it"], r["Wc"][:, i], linewidth=2, color=COL[lab], label=lab)
        ax.axhline(floors[kind], color="k", linestyle=":", linewidth=1.5, label="floor")
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(200, MAXIT)
        ax.set_xlabel("Iterations", fontsize=12)
        ax.set_title("coordinate {}  (sd {:.2f})".format(i + 1, refs[kind][:, i].std()), fontsize=12)
        ax.grid(alpha=0.3, which="both")
    axes[0].set_ylabel(r"$W_1$  (mean over replicas)", fontsize=12)
    axes[0].legend(fontsize=9)
    fig.suptitle(TARGETS[kind] + r"  —  $W_1$ per coordinate", fontsize=13)
    plt.tight_layout()
    plt.savefig("figures/uncon_cross_w1_percoord_{}.pdf".format(kind), dpi=300, bbox_inches="tight")
    plt.show()

## Experiment 2 — the targets

The 2-d marginal in dims 1 and 2, reference first.

In [ ]:
for kind in TARGETS:
    panels = [("target (exact i.i.d.)", refs[kind])] + [
        (lab, RES[kind][lab]["X"].reshape(-1, d)) for lab in MAIN]
    LIM = 3.2 * refs[kind][:, :2].std()
    fig, axes = plt.subplots(1, len(panels), figsize=(3.9 * len(panels), 4.2))
    shown = {}
    for j, (name, X) in enumerate(panels):
        ax = axes[j]
        shown[name] = density_panel(ax, X, LIM)
        ax.set_title(name, fontsize=11)
        ax.set_xlabel("$x_1$", fontsize=12)
        ax.set_aspect("equal", adjustable="box"); ax.grid(alpha=0.25)
        if j == 0:
            ax.set_ylabel("$x_2$", fontsize=12)
    fig.suptitle(TARGETS[kind] + "  —  the target, and how each scheme fills it",
                 fontsize=14, y=1.03)
    plt.tight_layout()
    plt.savefig("figures/uncon_cross_2d_{}.pdf".format(kind), dpi=300, bbox_inches="tight")
    plt.show()
    print("{}: fraction of walkers inside the plotted window  {}".format(
        kind, {k: round(v, 4) for k, v in shown.items()}))

## Experiment 3 — head to head

In [ ]:
HEADROOM = 1.5     # threshold must clear the scheme's own stationary error by this factor

def tau_cell(r, kind, m, base=None):
    # tau at m x floor.  It is a rate only if the threshold sits well above the scheme's OWN
    # stationary error; otherwise the curve keeps re-crossing and tau measures noise.
    if m * floors[kind] < HEADROOM * r["stat"].mean():
        return "too close"
    t = boot(r["taus"][m])
    if not np.isfinite(t[0]):
        return "too close"
    if base is None:
        return "{:.0f} [{:.0f}, {:.0f}]".format(*t)
    return "{:.0f}  ({})".format(t[0], fmt_speed(boot_speedup(base, r["taus"][m])).split(" [")[0])

for kind in TARGETS:
    base = RES[kind]["J = 0"]["taus"]
    rows = []
    for lab in MAIN:
        r = RES[kind][lab]
        ks = np.mean([max(ks_2samp(refs[kind][:, i], r["X"][j, :, i]).statistic for i in range(d))
                      for j in range(REPLICAS)])
        row = {"scheme": lab.replace("$", "")}
        for m in MULT:
            row["tau ({:g}x floor)".format(m)] = tau_cell(
                r, kind, m, None if r["s"] == 0 else base[m])
        row.update({"stationary W1": "{:.4f} [{:.4f}, {:.4f}]".format(*boot(r["stat"])),
                    "max KS": round(float(ks), 4),
                    "mean ||x||": round(r["rad"]["mean_r"], 3),
                    "sd ratio to exact": np.round(r["rad"]["sd"] / refs[kind].std(0), 3)})
        rows.append(row)
    row = {"scheme": "target (truth)"}
    for m in MULT:
        row["tau ({:g}x floor)".format(m)] = "--"
    rt = radial_stats(refs[kind])
    row.update({"stationary W1": "{:.4f}  (floor)".format(floors[kind]), "max KS": 0.0,
                "mean ||x||": round(rt["mean_r"], 3),
                "sd ratio to exact": np.round(np.ones(d), 3)})
    rows.append(row)
    print("\n### {}   (J = 0 columns are absolute; the rest are tau, with the speed-up in brackets)"
          .format(TARGETS[kind]))
    display(pd.DataFrame(rows).set_index("scheme"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))
for col, kind in enumerate(TARGETS):
    tm = [boot(RES[kind][l]["taus"][3.0]) for l in MAIN]
    st = [boot(RES[kind][l]["stat"]) for l in MAIN]
    xs_ = np.arange(len(MAIN))
    axes[0].errorbar(xs_ + col * 0.12, [v[0] for v in tm],
                     yerr=[[v[0] - v[1] for v in tm], [v[2] - v[0] for v in tm]],
                     fmt="o-", capsize=3, linewidth=2, label=TARGETS[kind].split(":")[0])
    axes[1].errorbar(xs_ + col * 0.12, [v[0] for v in st],
                     yerr=[[v[0] - v[1] for v in st], [v[2] - v[0] for v in st]],
                     fmt="o-", capsize=3, linewidth=2, label=TARGETS[kind].split(":")[0])
    axes[1].axhline(floors[kind], linestyle=":", linewidth=1.5,
                    color=["k", "0.55"][col], label="{} floor".format(kind))
for ax, ylab, title in [(axes[0], r"$\tau$  (iterations to $3\times$ floor)", "speed"),
                        (axes[1], r"stationary mean $W_1$", "accuracy")]:
    ax.set_xticks(np.arange(len(MAIN)))
    ax.set_xticklabels([l.replace("$", "") for l in MAIN], fontsize=10)
    ax.set_ylabel(ylab, fontsize=12); ax.set_title(title, fontsize=12)
    ax.grid(alpha=0.3); ax.legend(fontsize=9)
fig.suptitle(r"$J_s$ against $J=0$ without the constraint: speed and accuracy", fontsize=13)
plt.tight_layout()
plt.savefig("figures/uncon_cross_summary.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Experiment 4 — where the boundary atom went

On the ball the discretisation artefact was a mass atom on $\partial K$: a step near the wall gets its
normal component clipped by the projection, so mass piles up there. It was flat in $s$ — as it had to
be, since $J_s$ is exactly tangential and adds no normal drift — and it scaled like $\sqrt\eta$.

There is no wall here, so that atom does not exist. But the mechanism behind it has an unconstrained
shadow, and it is *not* flat in $s$. The skew drift is orthogonal to $x$, so it moves a walker along
the sphere through $x$ — to **first** order. A finite step of size $a\perp x$ takes
$\lVert x\rVert\mapsto\sqrt{\lVert x\rVert^2+\lVert a\rVert^2}$, which is larger. With
$\lVert a\rVert=\eta s\lVert x\rVert\lVert g\rVert$ that is an outward push of order
$\eta^2s^2\lVert x\rVert\lVert g\rVert^2$ per step. On the ball the projection absorbed it into the
atom. Without the wall it should show up as **radial inflation** — the cloud sitting further out than
$\pi$ does, with inflated coordinate variances.

This cell measures it directly, and avoids confounding it with the convergence transient by **starting
the walkers at exact draws from $\pi$**. At $s=0$, $\eta\to0$ the cloud would stay put; anything that
moves is discretisation bias. Sweeping $s$ and $\eta$ gives the scaling.

In [ ]:
# -----------------------------
# Start AT stationarity and measure the drift.  No transient to subtract.
# -----------------------------
REPLICAS_B = 4          # moments converge far faster in N than W1 does
STAT_STEPS = 40000
ETA_DIV    = [1, 2, 4]

@torch.no_grad()
def run_from_stationarity(kind, s, ref, eta_, n_steps=STAT_STEPS, S=REPLICAS_B, seed=SEED):
    U, U0, gU0 = build_target(kind)
    rng_loc = np.random.default_rng(seed + 17)
    x = torch.tensor(sample_target(kind, S * N, rng_loc))     # exact draws from pi
    torch.manual_seed(seed)
    for _ in range(n_steps):
        Delta = U(x) - U0(x)
        g = gU0(x) * torch.exp(Delta).unsqueeze(1)
        x = (x - eta_ * (g + Js(s, x, g))
             + np.sqrt(2 * eta_) * torch.exp(0.5 * Delta).unsqueeze(1) * torch.randn_like(x))
        if not torch.isfinite(x).all():
            print("      !! non-finite state (s = {:g}, eta = {:.1e}) -- diverged".format(
                s, eta_), flush=True)
            break
    Xn = x.numpy().reshape(S, N, d)
    Xf = Xn.reshape(-1, d)
    w1 = float(np.mean([[wasserstein_distance(ref[:, i], Xn[r, :, i]) for i in range(d)]
                        for r in range(S)]))
    return {"w1": w1, "mean_r": float(np.linalg.norm(Xf, axis=1).mean()),
            "sd": Xf.std(0), "X": Xf}

BIAS = {}
for kind in TARGETS:
    BIAS[kind] = {}
    r_exact = float(np.linalg.norm(refs[kind], axis=1).mean())
    sd_exact = refs[kind].std(0)
    print("\n### {}   exact mean ||x|| = {:.4f}".format(kind, r_exact), flush=True)
    for c in ETA_DIV:
        for s, lab in GRID:
            t0 = time.time()
            out = cached("bias_{}_{}_{:g}".format(kind, c, s),
                         lambda kind=kind, s=s, c=c:
                             run_from_stationarity(kind, s, refs[kind], eta / c))
            out["infl"] = out["mean_r"] / r_exact - 1.0
            out["sd_infl"] = float(np.mean(out["sd"] / sd_exact) - 1.0)
            out["c"], out["s"] = c, s
            BIAS[kind][(c, s)] = out
            print("   eta/{}  {:<12}  mean||x|| {:.4f}  inflation {:+.3%}  "
                  "sd inflation {:+.3%}  W1 {:.4f}   [{:.0f}s]".format(
                      c, lab.replace("$", ""), out["mean_r"], out["infl"],
                      out["sd_infl"], out["w1"], time.time() - t0), flush=True)

In [ ]:
# Does the inflation scale the way the mechanism predicts?
# The per-step outward push is ~ eta^2 s^2 ||x|| ||g||^2, and the restoring drift relaxes at
# rate ~ eta, so the stationary inflation attributable to the field should go like eta * s^2.
rows = []
for kind in TARGETS:
    for c in ETA_DIV:
        base = BIAS[kind][(c, 0.0)]["infl"]
        for s, lab in GRID:
            b = BIAS[kind][(c, s)]
            rows.append({"target": kind, "eta": eta / c, "s": s,
                         "radial inflation": "{:+.3%}".format(b["infl"]),
                         "field part (minus s=0)": "{:+.3%}".format(b["infl"] - base),
                         "/(eta s^2)": ("--" if s == 0 else
                                        round((b["infl"] - base) / ((eta / c) * s ** 2), 3)),
                         "stationary W1": round(b["w1"], 4),
                         "floor": round(floors[kind], 4)})
display(pd.DataFrame(rows).set_index(["target", "eta", "s"]))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))
for ax, kind in zip(axes, TARGETS):
    for s, lab in GRID:
        xs_e = np.array([eta / c for c in ETA_DIV])
        ys_e = np.array([BIAS[kind][(c, s)]["infl"] * 100 for c in ETA_DIV])
        ax.plot(xs_e, ys_e, "o-", color=COL[lab], linewidth=2, markersize=8, label=lab)
    ax.set_xscale("log"); ax.set_xlabel(r"$\eta$", fontsize=12)
    ax.set_ylabel(r"radial inflation  $E\|x\|/E\|x\|_\pi - 1$   (%)", fontsize=11)
    ax.axhline(0, color="k", linewidth=1)
    ax.set_title(TARGETS[kind], fontsize=11)
    ax.grid(alpha=0.3); ax.legend(fontsize=9)
fig.suptitle("The unconstrained shadow of the boundary atom: a tangential field is only "
             "tangential to first order", fontsize=13)
plt.tight_layout()
plt.savefig("figures/uncon_radial_inflation.pdf", dpi=300, bbox_inches="tight")
plt.show()

### Experiment 4b — the escape radius, and why $s=16$ is not usable here

Experiment 4 measures the inflation as a bulk average. The tail is worse, and it is worth doing the
arithmetic because it turns into a usable design rule.

At large $\lVert x\rVert$, per step,

$$\underbrace{-2\eta\,x\!\cdot\!g}_{\text{potential pulls in},\ \sim-2\eta c\lVert x\rVert}
\ +\ \underbrace{\eta^2s^2\lVert x\times g\rVert^2}_{\text{skew step pushes out},\ \sim+\eta^2s^2c^2\lVert x\rVert^2}$$

with $c=\lVert g\rVert$. The skew term is exactly tangential so it contributes nothing at first order;
this is the $\lVert a\rVert^2$ in $\lVert x\rVert^2\mapsto\lVert x\rVert^2+\lVert a\rVert^2$. The two
balance at

$$\boxed{\ r^\*=\frac{2}{\eta s^2 c}\ }$$

and beyond $r^\*$ the outward term wins, so a walker there is pushed further out, which strengthens the
push. Since $\lVert g\rVert=\lVert \hat y M^{-1}\rVert$ lies between the extreme singular values of
$M^{-1}$, $c\in[0.47,\,2.77]$ here, giving $r^\*\in[23,133]$ at $s=8$ and $r^\*\in[5.7,33]$ at $s=16$.
Target A has $q_{99.9}\lVert x\rVert=18.2$. **At $s=16$ a substantial part of the distribution sits
beyond $r^\*$.**

On the ball this could never bite: $R=0.5$ is far inside every $r^\*$ in the table, which is why the
constrained study found the field free in accuracy. The projection was not merely absorbing the
artefact into the atom — it was holding the walkers inside the radius where the scheme is stable at all.

Note $r^\*\propto1/\eta$, so refining the stepsize buys *validity* here, not just accuracy. That changes
what Experiment 5 is measuring.

In [ ]:
# -----------------------------
# Is the tail stationary, or still moving?  Cheap: 5000 walkers, one chain per setting.
# A converged chain has max||x|| flat between 40k and 80k; an escaping one keeps growing.
# -----------------------------
SV = np.linalg.svd(np.linalg.inv(L3), compute_uv=False)
print("singular values of M^-1:", np.round(SV, 3), " -> c = ||g|| in [{:.2f}, {:.2f}]".format(
    SV[-1], SV[0]))
print("r* = 2 / (eta s^2 c)\n")

def r_star(s, eta_):
    if s == 0:
        return (np.inf, np.inf)
    return (2 / (eta_ * s ** 2 * SV[0]), 2 / (eta_ * s ** 2 * SV[-1]))

N_S = 5000
@torch.no_grad()
def tail_probe(kind, s, eta_, n_steps=80000, seed=SEED):
    U, U0, gU0 = build_target(kind)
    torch.manual_seed(seed)
    x = torch.full((N_S, d), x_start)
    marks = {}
    for k in range(1, n_steps + 1):
        Delta = U(x) - U0(x)
        g = gU0(x) * torch.exp(Delta).unsqueeze(1)
        x = (x - eta_ * (g + Js(s, x, g))
             + np.sqrt(2 * eta_) * torch.exp(0.5 * Delta).unsqueeze(1) * torch.randn_like(x))
        if k in (n_steps // 2, n_steps):
            marks[k] = x.norm(dim=1).numpy().copy()
        if not torch.isfinite(x).all():
            for kk in (n_steps // 2, n_steps):
                marks.setdefault(kk, x.norm(dim=1).numpy().copy())
            print("      !! non-finite at {} (s={:g}, eta={:.1e})".format(k, s, eta_), flush=True)
            break
    return marks

STAB = {}
S_STAB = [0.0, 2.0, 4.0, 8.0, 16.0]
rows = []
for kind in TARGETS:
    ex = np.linalg.norm(refs[kind], axis=1)
    q999_ex = float(np.quantile(ex, 0.999))
    for s in S_STAB:
        m = cached("stab_{}_{:g}_1".format(kind, s),
                   lambda kind=kind, s=s: tail_probe(kind, s, eta))
        STAB[(kind, s, 1)] = m
        half, full = sorted(m)[0], sorted(m)[1]
        lo, hi = r_star(s, eta)
        rows.append({"target": kind, "s": s,
                     "r* (low c)": "inf" if not np.isfinite(hi) else round(hi, 1),
                     "r* (high c)": "inf" if not np.isfinite(lo) else round(lo, 1),
                     "q99.9 exact": round(q999_ex, 2),
                     "q99.9 @40k": round(float(np.quantile(m[half], 0.999)), 2),
                     "q99.9 @80k": round(float(np.quantile(m[full], 0.999)), 2),
                     "max @40k": round(float(m[half].max()), 2),
                     "max @80k": round(float(m[full].max()), 2),
                     "still growing?": "YES" if m[full].max() > 1.15 * m[half].max() else "no"})
display(pd.DataFrame(rows).set_index(["target", "s"]))
print("A chain that has found its stationary tail has max||x|| roughly flat between 40k and 80k.")
print("One that is escaping keeps climbing, and the column says so.")

In [ ]:
# does refining eta restore stability?  r* scales like 1/eta, so it should.
rows = []
for kind in TARGETS:
    q999_ex = float(np.quantile(np.linalg.norm(refs[kind], axis=1), 0.999))
    for c in [1, 2, 4]:
        m = cached("stab_{}_16_{}".format(kind, c),
                   lambda kind=kind, c=c: tail_probe(kind, S_TOP, eta / c))
        STAB[(kind, S_TOP, c)] = m
        half, full = sorted(m)[0], sorted(m)[1]
        lo, hi = r_star(S_TOP, eta / c)
        rows.append({"target": kind, "eta": "{:.1e}".format(eta / c),
                     "r* (low c)": round(hi, 1), "r* (high c)": round(lo, 1),
                     "q99.9 exact": round(q999_ex, 2),
                     "q99.9 @80k": round(float(np.quantile(m[full], 0.999)), 2),
                     "max @40k": round(float(m[half].max()), 2),
                     "max @80k": round(float(m[full].max()), 2),
                     "still growing?": "YES" if m[full].max() > 1.15 * m[half].max() else "no"})
display(pd.DataFrame(rows).set_index(["target", "eta"]))
print("s = {:g} at three stepsizes.  Same iteration count each time, so this is a question about".format(S_TOP))
print("whether the tail is stationary, not about how far the chain has run in continuous time.")

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4))
for ax, kind in zip(axes, TARGETS):
    ex = np.linalg.norm(refs[kind], axis=1)
    bins = np.linspace(0, max(30, 1.2 * ex.max()), 70)
    ax.hist(ex, bins=bins, histtype="step", linewidth=2.2, color="k",
            density=True, label="exact")
    for s in [v for v in (0.0, 8.0, 16.0) if (kind, v, 1) in STAB]:
        m = STAB[(kind, s, 1)]
        full = sorted(m)[1]
        ax.hist(m[full], bins=bins, histtype="step", linewidth=1.8, density=True,
                color=COL["J = 0" if s == 0 else "$J_s$, s={:g}".format(s)],
                label="s = {:g}".format(s))
        lo, hi = r_star(s, eta)
        if np.isfinite(hi) and hi < bins[-1]:
            ax.axvline(hi, linestyle="--", linewidth=1.2,
                       color=COL["$J_s$, s={:g}".format(s)])
    ax.set_yscale("log"); ax.set_xlabel(r"$\|x\|_2$", fontsize=12)
    ax.set_ylabel("density (log)", fontsize=11)
    ax.set_title(TARGETS[kind] + r"   (dashed: $r^*$ at $c=\sigma_{\max}$)", fontsize=11)
    ax.grid(alpha=0.3); ax.legend(fontsize=9)
fig.suptitle(r"Radial law: the skew field grows a tail the target does not have", fontsize=13)
plt.tight_layout()
plt.savefig("figures/uncon_escape_radius.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Experiment 5 — the stepsize trade

Same question as on the ball: refining $\eta$ by $c$ costs roughly $c\times$ the iterations, and buys
accuracy. Is there a $c$ at which $J_s$ still converges in fewer iterations than $J=0$ does at the
coarse stepsize, while being more accurate?

What accuracy means has changed. On the ball the currency was the boundary atom, flat in $s$, so the
field's speed-up was pure profit and the only question was arithmetic. Here the currency is the
stationary $W_1$ and the radial inflation of Experiment 4 — and that one **grows** with $s$, so the
field is no longer free and the trade has a second term working against it.

In [ ]:
REFINE = [2, 4]
RES_R = {}
for kind in TARGETS:
    RES_R[kind] = {}
    for c in REFINE:
        lab = "{}, $\\eta$/{}".format(TOP, c)
        t0 = time.time()
        def _go(kind=kind, c=c):
            it, W, Wc, X = run_replicas(kind, S_TOP, refs[kind],
                                        n_steps=MAXIT * c, eta=eta / c, track=TRACK * c)
            return {"it": it, "W": W, "Wc": Wc, "X": X, "s": S_TOP, "c": c,
                    "taus": {m: np.array([tau_one(it, w, m * floors[kind]) for w in W])
                             for m in MULT},
                    "stat": W[:, -10:].mean(axis=1),
                    "rad": radial_stats(X.reshape(-1, d))}
        RES_R[kind][lab] = cached("refine_{}_{}".format(kind, c), _go)
        print("   {:<24} {:5.0f}s".format(kind + "  eta/" + str(c), time.time() - t0), flush=True)

In [ ]:
for kind in TARGETS:
    base = RES[kind]["J = 0"]["taus"]
    r_exact = float(np.linalg.norm(refs[kind], axis=1).mean())
    rows = []
    entries = [("$J = 0$", RES[kind]["J = 0"], 1), (TOP, RES[kind][TOP], 1)]
    entries += [(k, v, v["c"]) for k, v in RES_R[kind].items()]
    for lab, r, c in entries:
        ok = (3.0 * floors[kind] >= HEADROOM * r["stat"].mean()
              and np.any(np.isfinite(r["taus"][3.0])))
        rows.append({"scheme": lab.replace("$", "").replace("\\eta", "eta"),
                     "eta": "{:.1e}".format(eta / c),
                     "tau (3x floor)": fmt_tau(boot(r["taus"][3.0])) if ok else "too close",
                     "vs J=0 iterations": "--" if lab == "$J = 0$" else (
                         "{:.2f}x".format(np.nanmean(base[3.0]) / np.nanmean(r["taus"][3.0]))
                         if ok else "--"),
                     "stationary W1": "{:.4f} [{:.4f}, {:.4f}]".format(*boot(r["stat"])),
                     "radial inflation": "{:+.2%}".format(r["rad"]["mean_r"] / r_exact - 1)})
    rows.append({"scheme": "target (truth)", "eta": "--", "tau (3x floor)": "--",
                 "vs J=0 iterations": "--",
                 "stationary W1": "{:.4f}  (floor)".format(floors[kind]),
                 "radial inflation": "0.00%"})
    print("\n### {}".format(TARGETS[kind]))
    display(pd.DataFrame(rows).set_index("scheme"))

In [ ]:
# the dominance picture: down and to the left beats J = 0 on both axes at once
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))
for ax, kind in zip(axes, TARGETS):
    pts = [("$J = 0$", RES[kind]["J = 0"], "#C44E52", "o"),
           (TOP, RES[kind][TOP], "#55A868", "o")]
    pts += [(k, v, "#4C72B0", "s") for k, v in RES_R[kind].items()]
    for lab, r, c, mk in pts:
        t = boot(r["taus"][3.0])
        if not np.isfinite(t[0]):
            continue
        ax.errorbar(t[0], r["stat"].mean(), xerr=[[t[0] - t[1]], [t[2] - t[0]]],
                    fmt=mk, color=c, markersize=10, capsize=4, linewidth=2)
        ax.annotate(lab, (t[0], r["stat"].mean()), textcoords="offset points",
                    xytext=(9, 6), fontsize=9)
    j0 = boot(RES[kind]["J = 0"]["taus"][3.0])
    if np.isfinite(j0[0]):
        ax.axvline(j0[0], color="#C44E52", linestyle="--", linewidth=1.2)
    ax.axhline(RES[kind]["J = 0"]["stat"].mean(), color="#C44E52", linestyle="--", linewidth=1.2)
    ax.axhline(floors[kind], color="k", linestyle=":", linewidth=1.6, label="sampling floor")
    ax.set_xlabel(r"$\tau$   (iterations to $3\times$ floor)", fontsize=12)
    ax.set_ylabel(r"stationary mean $W_1$", fontsize=12)
    ax.set_title(TARGETS[kind], fontsize=12)
    ax.grid(alpha=0.3); ax.legend(fontsize=9, loc="upper right")
fig.suptitle("Spending the speed-up: everything below and left of the dashed lines beats "
             r"$J=0$ on both axes", fontsize=13)
plt.tight_layout()
plt.savefig("figures/uncon_cross_tradeoff.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
SHOW = [("target (exact)", None), ("$J = 0$", "J = 0")] + [(l, l) for l in MAIN[1:]]
fig, axes = plt.subplots(2, len(SHOW), figsize=(3.6 * len(SHOW), 7.6))
for row, kind in enumerate(TARGETS):
    LIM = 3.2 * refs[kind][:, :2].std()
    for col, (name, key) in enumerate(SHOW):
        X = refs[kind] if key is None else (
            RES[kind][key]["X"] if key in RES[kind] else RES_R[kind][key]["X"]).reshape(-1, d)
        ax = axes[row, col]
        density_panel(ax, X, LIM)
        ax.set_aspect("equal", adjustable="box")
        ax.set_title(name, fontsize=10)
        ax.set_xlabel("$x_1$", fontsize=11)
        ax.set_ylabel("$x_2$" if col == 0 else "", fontsize=11)
        ax.grid(alpha=0.25)
    axes[row, 0].text(-0.42, 0.5, TARGETS[kind].split(":")[0], transform=axes[row, 0].transAxes,
                      rotation=90, va="center", ha="center", fontsize=12)
fig.suptitle("Both targets, unconstrained: 2-d marginals", fontsize=14, y=1.01)
plt.tight_layout()
fig.subplots_adjust(hspace=0.5)
plt.savefig("figures/uncon_cross_2d_both.pdf", dpi=300, bbox_inches="tight")
plt.show()